# Pandas Technical Deep Dive - Advanced Data Manipulation Techniques

This notebook demonstrates advanced pandas techniques for data analysis including:

- Data generation and setup
- Data cleaning with replacing and thresholding
- Intelligent imputation strategies
- Function application techniques (apply, map, vectorized)
- Grouping operations and aggregations
- Time-based analysis
- Data merging strategies
- MultiIndex operations
- Stacking and unstacking
- Pivot tables with margins
- Advanced analytics and visualization

Let's start by importing the required libraries and setting up our environment.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("Pandas Technical Deep Dive Demo")
print("=" * 50)

## Section 1: Data Generation and Setup

We'll create a comprehensive simulated retail dataset with various product categories, sales representatives, regions, and customer information. This dataset will include intentional data quality issues to demonstrate cleaning techniques.

In [ ]:
print("\n1. GENERATING SIMULATED RETAIL DATASET")
print("-" * 40)

# Generate dates
start_date = datetime(2023, 1, 1)
end_date = datetime(2023, 12, 31)
date_range = pd.date_range(start_date, end_date, freq='D')

# Product categories and names
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Books']
products = {
    'Electronics': ['Laptop', 'Phone', 'Tablet', 'Headphones', 'Camera'],
    'Clothing': ['T-Shirt', 'Jeans', 'Dress', 'Jacket', 'Shoes'],
    'Home & Garden': ['Sofa', 'Table', 'Plant', 'Lamp', 'Rug'],
    'Sports': ['Basketball', 'Tennis Racket', 'Yoga Mat', 'Dumbbells', 'Bike'],
    'Books': ['Fiction', 'Biography', 'Science', 'History', 'Cookbook']
}

# Sales representatives and regions
sales_reps = ['Alice Johnson', 'Bob Smith', 'Charlie Brown', 'Diana Lee', 'Eve Wilson']
regions = ['North', 'South', 'East', 'West', 'Central']

print(f"Categories: {categories}")
print(f"Sales Reps: {sales_reps}")
print(f"Regions: {regions}")

In [ ]:
# Generate sales data
n_records = 10000
sales_data = []

for _ in range(n_records):
    category = np.random.choice(categories)
    product = np.random.choice(products[category])
    date = np.random.choice(date_range)
    rep = np.random.choice(sales_reps)
    region = np.random.choice(regions)
    
    # Base price varies by category
    base_prices = {
        'Electronics': (200, 1500),
        'Clothing': (20, 200),
        'Home & Garden': (50, 800),
        'Sports': (15, 500),
        'Books': (10, 50)
    }
    
    price = np.random.uniform(*base_prices[category])
    quantity = np.random.randint(1, 10)
    
    # Add some seasonality and trends
    month_factor = 1 + 0.1 * np.sin(2 * np.pi * date.month / 12)
    price *= month_factor
    
    # Customer info
    customer_id = np.random.randint(1000, 9999)
    customer_age = np.random.randint(18, 80)
    
    sales_data.append({
        'date': date,
        'category': category,
        'product': product,
        'price': round(price, 2),
        'quantity': quantity,
        'sales_amount': round(price * quantity, 2),
        'sales_rep': rep,
        'region': region,
        'customer_id': customer_id,
        'customer_age': customer_age
    })

# Create DataFrame
df = pd.DataFrame(sales_data)

print(f"Generated dataset with {len(df)} records")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

In [ ]:
# Introduce some data quality issues for demonstration
# Add missing values
missing_indices = np.random.choice(df.index, size=int(0.05 * len(df)), replace=False)
df.loc[missing_indices[:len(missing_indices)//2], 'price'] = np.nan
df.loc[missing_indices[len(missing_indices)//2:], 'customer_age'] = np.nan

# Add some outliers
outlier_indices = np.random.choice(df.index, size=50, replace=False)
df.loc[outlier_indices, 'price'] *= 10  # Extreme prices

# Add some inconsistent data
df.loc[np.random.choice(df.index, 20), 'region'] = 'NORTH'  # Inconsistent case
df.loc[np.random.choice(df.index, 15), 'region'] = 'north'

print("First few records:")
print(df.head())

print("\nDataset info:")
print(df.info())

## Section 2: Data Cleaning - Replacing and Thresholding

Now we'll clean our data by addressing inconsistencies and outliers using pandas replacement and thresholding techniques.

In [ ]:
print("2. DATA CLEANING - REPLACING AND THRESHOLDING")
print("-" * 50)

# Show data quality issues
print("Data Quality Assessment:")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nUnique regions: {df['region'].unique()}")
print(f"Price statistics:\n{df['price'].describe()}")

In [ ]:
# Fix inconsistent region names
print("Fixing inconsistent region names...")
region_mapping = {'NORTH': 'North', 'north': 'North', 'SOUTH': 'South'}
df['region'] = df['region'].replace(region_mapping)
print(f"Fixed regions: {df['region'].unique()}")

# Apply price thresholds to handle outliers
print("\nApplying price thresholds...")
# Calculate category-wise price percentiles
category_thresholds = df.groupby('category')['price'].quantile([0.01, 0.99]).unstack()
print("Category price thresholds (1st and 99th percentile):")
print(category_thresholds)

In [ ]:
# Apply thresholds
for category in df['category'].unique():
    if category in category_thresholds.index:
        mask = df['category'] == category
        lower_threshold = category_thresholds.loc[category, 0.01]
        upper_threshold = category_thresholds.loc[category, 0.99]
        
        # Cap extreme values
        df.loc[mask & (df['price'] < lower_threshold), 'price'] = lower_threshold
        df.loc[mask & (df['price'] > upper_threshold), 'price'] = upper_threshold

print("Applied price thresholds to control outliers")
print("Updated price statistics:")
print(df['price'].describe())

## Section 3: Intelligent Imputation

We'll demonstrate different imputation strategies for handling missing values in our dataset.

In [ ]:
print("3. INTELLIGENT IMPUTATION")
print("-" * 30)

print("Before imputation:")
print(f"Missing prices: {df['price'].isnull().sum()}")
print(f"Missing customer ages: {df['customer_age'].isnull().sum()}")

# Impute prices using category-based means
df['price'] = df.groupby('category')['price'].transform(
    lambda x: x.fillna(x.mean())
)

# Impute customer ages using forward fill (assuming temporal ordering)
df = df.sort_values('date')
df['customer_age'] = df['customer_age'].fillna(method='ffill')

# Recalculate sales amount after price imputation
df['sales_amount'] = df['price'] * df['quantity']

print("\nAfter imputation:")
print(f"Missing prices: {df['price'].isnull().sum()}")
print(f"Missing customer ages: {df['customer_age'].isnull().sum()}")

## Section 4: Function Application Techniques

We'll compare different approaches for applying functions to pandas data: vectorized operations, map, and apply methods.

In [ ]:
print("4. FUNCTION APPLICATION TECHNIQUES")
print("-" * 40)

# Create customer segments using different methods
print("Creating customer segments using different approaches...")

# Method 1: Vectorized operations (fastest)
print("1. Vectorized approach:")
df['age_segment_vectorized'] = np.where(
    df['customer_age'] < 30, 'Young',
    np.where(df['customer_age'] < 50, 'Middle', 'Senior')
)

# Method 2: Map method
print("2. Map approach:")
age_mapping = {age: 'Young' if age < 30 else 'Middle' if age < 50 else 'Senior' 
               for age in range(18, 81)}
df['age_segment_mapped'] = df['customer_age'].map(age_mapping)

# Method 3: Apply method
print("3. Apply approach:")
def categorize_age(age):
    if age < 30:
        return 'Young'
    elif age < 50:
        return 'Middle'
    else:
        return 'Senior'

df['age_segment_applied'] = df['customer_age'].apply(categorize_age)

# Verify all methods produce same result
print(f"All methods equal: {df['age_segment_vectorized'].equals(df['age_segment_applied'])}")

In [ ]:
# Performance comparison example (using apply for complex business logic)
print("Creating complex customer value score using apply...")

def calculate_customer_value_score(row):
    """Complex business logic for customer scoring"""
    base_score = row['sales_amount'] / 100
    
    # Age bonus
    if row['customer_age'] < 30:
        age_bonus = 1.2
    elif row['customer_age'] < 50:
        age_bonus = 1.1
    else:
        age_bonus = 1.0
    
    # Category bonus
    category_bonus = {'Electronics': 1.3, 'Clothing': 1.1}.get(row['category'], 1.0)
    
    return base_score * age_bonus * category_bonus

df['customer_value_score'] = df.apply(calculate_customer_value_score, axis=1)
print("Customer value scores calculated")
print(f"Score range: {df['customer_value_score'].min():.2f} - {df['customer_value_score'].max():.2f}")

## Section 5: Grouping Operations

Let's explore various grouping operations and aggregation techniques in pandas.

In [ ]:
print("5. GROUPING OPERATIONS")
print("-" * 30)

# Basic grouping
print("Basic grouping - Sales by category:")
category_sales = df.groupby('category')['sales_amount'].agg(['sum', 'mean', 'count'])
print(category_sales)

In [ ]:
# Multiple column grouping
print("Multiple column grouping - Sales by category and region:")
category_region_sales = df.groupby(['category', 'region'])['sales_amount'].sum().unstack(fill_value=0)
print(category_region_sales)

In [ ]:
# Advanced aggregation with custom functions
print("Advanced aggregation with multiple functions:")
advanced_agg = df.groupby('category').agg({
    'sales_amount': ['sum', 'mean', 'std'],
    'quantity': ['sum', 'mean'],
    'price': ['min', 'max'],
    'customer_value_score': 'mean'
})
print(advanced_agg)

In [ ]:
# Named aggregations (pandas 0.25+)
print("Named aggregations:")
named_agg = df.groupby('category').agg(
    total_sales=('sales_amount', 'sum'),
    avg_sales=('sales_amount', 'mean'),
    total_quantity=('quantity', 'sum'),
    unique_customers=('customer_id', 'nunique'),
    avg_customer_score=('customer_value_score', 'mean')
)
print(named_agg)

# Transform example - calculate rolling statistics
print("\nTransform example - within-group standardization:")
df['sales_amount_standardized'] = df.groupby('category')['sales_amount'].transform(
    lambda x: (x - x.mean()) / x.std()
)
print(f"Standardized sales amount created (mean=0, std=1 within each category)")

## Section 6: Time-Based Analysis with Grouping

We'll create time-based features and perform temporal analysis on our sales data.

In [ ]:
print("6. TIME-BASED GROUPING ANALYSIS")
print("-" * 40)

# Create time-based features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df['day_of_week'] = df['date'].dt.day_name()

# Monthly sales trends
monthly_sales = df.groupby('month')['sales_amount'].sum()
print("Monthly sales trends:")
print(monthly_sales)

# Quarterly sales by category
quarterly_category = df.groupby(['quarter', 'category'])['sales_amount'].sum().unstack()
print("\nQuarterly sales by category:")
print(quarterly_category)

## Section 7: Data Merging Demonstration

We'll create additional datasets and demonstrate different types of joins and merges.

In [ ]:
print("7. DATA MERGING OPERATIONS")
print("-" * 35)

# Create additional datasets for merging demonstration
print("Creating additional datasets for merging...")

# Sales rep information
rep_info = pd.DataFrame({
    'sales_rep': sales_reps,
    'hire_date': pd.date_range('2020-01-01', periods=len(sales_reps), freq='90D'),
    'department': ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'General'],
    'salary': [55000, 52000, 58000, 51000, 54000]
})

print("Sales rep information:")
print(rep_info)

In [ ]:
# Product information
all_products = []
suppliers = ['SupplierA', 'SupplierB', 'SupplierC', 'SupplierD']
for cat, prods in products.items():
    for prod in prods:
        all_products.append({
            'category': cat,
            'product': prod,
            'cost_price': np.random.uniform(10, 100),
            'supplier': np.random.choice(suppliers),
            'launch_date': np.random.choice(pd.date_range('2020-01-01', '2023-06-01'))
        })

product_info = pd.DataFrame(all_products)
print("Product information:")
print(product_info.head())

In [ ]:
# Perform different types of merges
print("Demonstrating different merge types...")

# Inner merge - only records with matches in both datasets
inner_merged = pd.merge(df, rep_info, on='sales_rep', how='inner')
print(f"Inner merge result: {len(inner_merged)} records (original: {len(df)})")

# Left merge - all records from sales data, matched with rep info where available
left_merged = pd.merge(df, rep_info, on='sales_rep', how='left')
print(f"Left merge result: {len(left_merged)} records")

# Merge with product information
df_with_product = pd.merge(df, product_info, on=['category', 'product'], how='left')
print(f"Merged with product info: {len(df_with_product)} records")

In [ ]:
# Calculate profit margin
df_with_product['profit'] = df_with_product['sales_amount'] - (df_with_product['cost_price'] * df_with_product['quantity'])
df_with_product['profit_margin'] = (df_with_product['profit'] / df_with_product['sales_amount']) * 100

print("Profit analysis after merging:")
profit_by_category = df_with_product.groupby('category')['profit_margin'].mean()
print(profit_by_category)

## Section 8: MultiIndex Operations

We'll explore hierarchical indexing (MultiIndex) for advanced data organization and analysis.

In [ ]:
print("8. MULTIINDEX OPERATIONS")
print("-" * 30)

# Create MultiIndex DataFrame
print("Creating MultiIndex for hierarchical analysis...")

# Group by multiple levels and create MultiIndex
multi_df = df.groupby(['region', 'category', 'month']).agg({
    'sales_amount': ['sum', 'mean'],
    'quantity': 'sum',
    'customer_id': 'nunique'
}).round(2)

print("MultiIndex DataFrame structure:")
print(multi_df.head(10))

In [ ]:
# MultiIndex selection examples
print("MultiIndex selection examples:")
print("Sales for North region:")
north_sales = multi_df.loc['North']
print(north_sales.head())

print("\nElectronics sales across all regions:")
electronics_sales = multi_df.xs('Electronics', level='category')
print(electronics_sales.head())

# Cross-section analysis
print("\nCross-section: January sales across all regions and categories:")
jan_sales = multi_df.xs(1, level='month')
print(jan_sales.head())

## Section 9: Stacking and Unstacking

We'll demonstrate how to reshape data using stack and unstack operations.

In [ ]:
print("9. STACKING AND UNSTACKING")
print("-" * 35)

# Create a pivot for stacking/unstacking demonstration
region_month_sales = df.pivot_table(
    values='sales_amount',
    index='region',
    columns='month',
    aggfunc='sum',
    fill_value=0
)

print("Original pivot table (wide format):")
print(region_month_sales)

In [ ]:
# Stack: wide to long
stacked = region_month_sales.stack()
print(f"After stacking (long format): {stacked.shape}")
print(stacked.head(10))

# Unstack: long to wide
unstacked = stacked.unstack()
print(f"\nAfter unstacking back (wide format): {unstacked.shape}")
print(unstacked.head())

In [ ]:
# Advanced stacking with MultiIndex
multi_pivot = df.pivot_table(
    values=['sales_amount', 'quantity'],
    index='region',
    columns='category',
    aggfunc='sum',
    fill_value=0
)

print("Multi-value pivot table:")
print(multi_pivot.head())

# Stack specific level
stacked_level = multi_pivot.stack(level=0)  # Stack the values level
print(f"\nStacked by value type: {stacked_level.shape}")
print(stacked_level.head())

## Section 10: Pivoting with Margins

We'll create pivot tables with totals and margins for comprehensive analysis.

In [ ]:
print("10. PIVOTING WITH MARGINS")
print("-" * 35)

# Basic pivot table
basic_pivot = df.pivot_table(
    values='sales_amount',
    index='category',
    columns='quarter',
    aggfunc='sum',
    fill_value=0
)
print("Basic pivot - Sales by category and quarter:")
print(basic_pivot)

In [ ]:
# Pivot with margins (totals)
pivot_with_margins = df.pivot_table(
    values='sales_amount',
    index='category',
    columns='quarter',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='TOTAL'
)
print("Pivot with margins:")
print(pivot_with_margins)

In [ ]:
# Multi-level pivot with margins
complex_pivot = df.pivot_table(
    values=['sales_amount', 'quantity'],
    index=['category', 'region'],
    columns='quarter',
    aggfunc={'sales_amount': 'sum', 'quantity': 'mean'},
    fill_value=0,
    margins=True
)
print("Complex pivot with multiple values and margins:")
print(complex_pivot.head(15))

## Section 11: Advanced Analytics and Insights

We'll perform sophisticated analysis including customer segmentation and performance evaluation.

In [ ]:
print("11. ADVANCED ANALYTICS AND INSIGHTS")
print("-" * 45)

# Customer segmentation analysis
print("Customer Segmentation Analysis:")
customer_analysis = df.groupby('customer_id').agg({
    'sales_amount': ['sum', 'count', 'mean'],
    'date': ['min', 'max'],
    'category': lambda x: x.nunique()
}).round(2)

customer_analysis.columns = [f"{col[0]}_{col[1]}" for col in customer_analysis.columns]
customer_analysis['days_active'] = (customer_analysis['date_max'] - customer_analysis['date_min']).dt.days

# Customer value segmentation
customer_analysis['customer_segment'] = pd.cut(
    customer_analysis['sales_amount_sum'],
    bins=[0, 500, 1500, float('inf')],
    labels=['Low Value', 'Medium Value', 'High Value']
)

print("Customer segments distribution:")
print(customer_analysis['customer_segment'].value_counts())

In [ ]:
# Seasonal analysis
print("Seasonal Analysis:")
seasonal_analysis = df.groupby(['month', 'category'])['sales_amount'].sum().unstack()
seasonal_trends = seasonal_analysis.div(seasonal_analysis.sum(axis=1), axis=0) * 100  # Percentage contribution

print("Category contribution by month (%):")
print(seasonal_trends.round(1))

In [ ]:
# Sales rep performance analysis
print("Sales Rep Performance Analysis:")
rep_performance = df.groupby('sales_rep').agg({
    'sales_amount': ['sum', 'mean', 'count'],
    'customer_id': 'nunique',
    'category': lambda x: x.nunique()
}).round(2)

rep_performance.columns = [f"{col[0]}_{col[1]}" for col in rep_performance.columns]
rep_performance['avg_deal_size'] = rep_performance['sales_amount_sum'] / rep_performance['sales_amount_count']
rep_performance = rep_performance.sort_values('sales_amount_sum', ascending=False)

print(rep_performance)

## Section 12: Data Visualization

We'll create comprehensive visualizations to summarize our analysis findings.

In [ ]:
print("12. DATA VISUALIZATION SETUP")
print("-" * 40)

# Set up matplotlib
plt.style.use('default')
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Sales Data Analysis Dashboard', fontsize=16, fontweight='bold')

# Plot 1: Monthly sales trend
monthly_sales.plot(kind='line', ax=axes[0,0], color='blue', marker='o')
axes[0,0].set_title('Monthly Sales Trend')
axes[0,0].set_xlabel('Month')
axes[0,0].set_ylabel('Sales Amount')
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Sales by category
category_sales['sum'].plot(kind='bar', ax=axes[0,1], color='green')
axes[0,1].set_title('Total Sales by Category')
axes[0,1].set_xlabel('Category')
axes[0,1].set_ylabel('Total Sales')
axes[0,1].tick_params(axis='x', rotation=45)

# Plot 3: Regional performance
regional_sales = df.groupby('region')['sales_amount'].sum()
regional_sales.plot(kind='pie', ax=axes[1,0], autopct='%1.1f%%')
axes[1,0].set_title('Sales Distribution by Region')

# Plot 4: Customer age distribution
df['customer_age'].hist(bins=20, ax=axes[1,1], color='orange', alpha=0.7)
axes[1,1].set_title('Customer Age Distribution')
axes[1,1].set_xlabel('Age')
axes[1,1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print("Visualization plots generated successfully!")

## Section 13: Final Summary and Export

Let's summarize our analysis and prepare the data for further use.

In [ ]:
print("13. FINAL SUMMARY")
print("-" * 25)

print("Dataset Summary:")
print(f"Total records processed: {len(df):,}")
print(f"Date range: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")
print(f"Total sales amount: ${df['sales_amount'].sum():,.2f}")
print(f"Average deal size: ${df['sales_amount'].mean():.2f}")
print(f"Unique customers: {df['customer_id'].nunique():,}")
print(f"Categories analyzed: {df['category'].nunique()}")
print(f"Regions covered: {df['region'].nunique()}")
print(f"Sales representatives: {df['sales_rep'].nunique()}")

In [ ]:
print("Top Performers:")
print("Best performing category:", category_sales['sum'].idxmax(), f"(${category_sales['sum'].max():,.2f})")
print("Best performing region:", regional_sales.idxmax(), f"(${regional_sales.max():,.2f})")
print("Best performing rep:", rep_performance.index[0], f"(${rep_performance['sales_amount_sum'].iloc[0]:,.2f})")

print("\nKey Insights Generated:")
print("✓ Data cleaning and imputation completed")
print("✓ Customer segmentation analysis performed")
print("✓ Seasonal trends identified")
print("✓ Sales representative performance evaluated")
print("✓ Product profitability calculated")
print("✓ Multi-dimensional analysis with hierarchical indexing")

# Save processed data for assignment
df_clean = df.copy()
print(f"\nProcessed dataset ready for further analysis!")
print("Demo completed successfully!")

## Conclusion

This notebook has demonstrated advanced pandas techniques for data manipulation and analysis:

1. **Data Generation**: Created a realistic dataset with intentional quality issues
2. **Data Cleaning**: Applied replacing and thresholding techniques to handle inconsistencies and outliers
3. **Imputation**: Used intelligent strategies for filling missing values
4. **Function Application**: Compared vectorized, map, and apply approaches
5. **Grouping**: Performed complex aggregations and transformations
6. **Time Analysis**: Created temporal features and analyzed trends
7. **Merging**: Demonstrated various join types and data combination techniques
8. **MultiIndex**: Explored hierarchical indexing for advanced data organization
9. **Reshaping**: Used stack/unstack for data transformation
10. **Pivot Tables**: Created comprehensive summaries with margins
11. **Analytics**: Performed customer segmentation and performance analysis
12. **Visualization**: Generated insightful charts and plots

The cleaned dataset is now ready for further analysis, machine learning, or reporting purposes.

---

**Technical Deep Dive Completed Successfully!**